In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [3]:
data='https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

In [4]:
!curl -L -O $data


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0           00:33              0
  0      0   0      0   0      0      0      0           00:34              0
  0      0   0      0   0      0      0      0           00:35              0
  0      0   0      0   0      0      0      0           00:36              0
  0      0   0      0   0      0      0      0           00:37              0
  0      0   0      0   0      0      0      0           00:38              0
  0      0   0      0   0      0      0      0           00:39              0
 85 620.2k  85 528.0k   0      0  13327      0   00:47   00:40   00:07 104.3k
100 620.2k 100 620.2k   0      0  15597      0   00:40   00:40         104.3k
100 620.2k 100 620.2k   0      0  15597      0   00:40   00:40         104.3k
100 620.2k 100 620.2k   0      0  15597      0   00:40   00:40 

In [7]:
df= pd.read_csv('car_fuel_efficiency_2026.csv').fillna(0)

In [9]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [10]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [11]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

In [12]:
from sklearn.feature_extraction import DictVectorizer
train_dicts = df_train.drop(columns='fuel_efficiency_mpg').to_dict(orient='records')
val_dicts = df_val.drop(columns='fuel_efficiency_mpg').to_dict(orient='records')


In [13]:
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)  
X_val = dv.transform(val_dicts)       
features = list(dv.get_feature_names_out())


QUESTION 1

In [ ]:
from sklearn.tree import DecisionTreeRegressor
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)
split_feature = features[dt.tree_.feature[0]]
print(split_feature.split('=')[0])   

model_year


QUESTION 2

In [16]:
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
print(root_mean_squared_error(y_val, rf.predict(X_val)))

1.8332197358745623


QUESTION 3

In [17]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, rf.predict(X_val))
    print(n, round(rmse, 3))

10 1.833
50 1.774
100 1.77
150 1.769


QUESTION 4

In [18]:
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d,
                                   random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        scores.append(root_mean_squared_error(y_val, rf.predict(X_val)))
    print(d, np.mean(scores))

10 1.7543924943801008
15 1.7842243445385957
20 1.7867711878395158
25 1.7863054936840876


QUESTION 5

In [19]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20,
                           random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
importances = dict(zip(features, rf.feature_importances_))

candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
for f in sorted(candidates, key=lambda f: -importances[f]):
    print(f, importances[f])

vehicle_weight 0.20331343574711913
horsepower 0.07827124720421208
engine_displacement 0.06128144481858746
acceleration 0.054997632153652054


QUESTION 6

In [20]:
import xgboost as xgb

In [21]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {'eta': eta, 'max_depth': 6, 'min_child_weight': 1,
                  'objective': 'reg:squarederror', 'nthread': 8,
                  'seed': 1, 'verbosity': 1}
    results = {}
    xgb.train(xgb_params, dtrain, num_boost_round=100,
              evals=watchlist, evals_result=results, verbose_eval=False)
    val_rmse = results['val']['rmse']
    print(eta, val_rmse[-1], min(val_rmse), np.argmin(val_rmse))

0.3 1.8264579949460302 1.7320478064805087 10
0.1 1.7248100768778967 1.7121714763059768 39
